In [ ]:
# ── Cell 0: List all input files/folders to identify your CSVs + image folders ─
from pathlib import Path

print("=== All files under /kaggle/input ===")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_file() and p.suffix in [".csv", ".jpg", ".jpeg", ".png"]:
        if p.suffix == ".csv":
            print(f"[CSV]  {p}")
    elif p.is_dir():
        n_images = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.jpeg"))) + len(list(p.glob("*.png")))
        if n_images > 0:
            print(f"[DIR, {n_images} images]  {p}")

In [ ]:
# ── Cell 1: Configure CSV <-> image folder pairs ────────────────────────────
import pandas as pd
from pathlib import Path

# Each tuple = (path_to_csv, path_to_that_csv's_image_folder)
csv_folder_pairs = [
    (
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/human_queries_pilot_50images_known.csv"),
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/images_KNOWN")
    ),
    (
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/human_queries_pilot_50images_new.csv"),
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/new_images")
    ),
    (
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/human_query_known2.csv"),
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/images_know_2")
    ),
    (
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/human_queries_pilot_50images_new2.csv"),
        Path("/kaggle/input/datasets/deboparna9das/human-queries-files/new_images 2nd")
    ),
]

for csv_path, img_dir in csv_folder_pairs:
    print(f"{'✅' if csv_path.exists() else '❌'} {csv_path}")
    print(
        f"{'✅' if img_dir.exists() else '❌'} "
        f"{img_dir} ({len(list(img_dir.glob('*.jpg')))} jpgs)"
    )

In [ ]:
# ── Cell 2 (fixed): Combine all 4 CSVs, handling mixed encodings ─────────────
def read_csv_robust(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {path} with any known encoding")

all_dfs = []
counter = 1
for csv_path, _ in csv_folder_pairs:
    df = read_csv_robust(csv_path).reset_index(drop=True)
    df["query_id"] = [f"q{counter+i:05d}" for i in range(len(df))]
    counter += len(df)
    df["_source_csv"] = csv_path.name
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
assert combined["query_id"].duplicated().sum() == 0, "duplicate query_ids after merge!"

print(f"✅ Combined: {len(combined)} total queries, 0 duplicate ids")
print(combined["_source_csv"].value_counts())

In [ ]:
# ── Cell 3: Filter each source folder down to only images actually used ──────
import shutil

OUT_KNOWN = Path("/kaggle/working/human_eval_final/known")
OUT_NEW   = Path("/kaggle/working/human_eval_final/new")
OUT_KNOWN.mkdir(parents=True, exist_ok=True)
OUT_NEW.mkdir(parents=True, exist_ok=True)

def find_image_file(img_dir, img_id):
    for ext in [".jpg", ".jpeg", ".png"]:
        candidates = list(img_dir.rglob(f"{img_id}{ext}"))
        if candidates:
            return candidates[0]
    return None

copied, missing = [], []
for csv_path, img_dir in csv_folder_pairs:
    sub = combined[combined["_source_csv"] == csv_path.name]
    for _, row in sub.iterrows():
        img_id = int(row["image_id"])
        pool = row["image_pool"]  # "known" or "new"
        dest_dir = OUT_KNOWN if pool == "known" else OUT_NEW
        dest_path = dest_dir / f"{img_id}.jpg"
        if dest_path.exists():
            continue  # already copied (shared image across multiple rows)
        src = find_image_file(img_dir, img_id)
        if src:
            shutil.copy(src, dest_path)
            copied.append(img_id)
        else:
            missing.append((img_id, csv_path.name))

print(f"✅ Copied {len(copied)} unique images")
print(f"   known/: {len(list(OUT_KNOWN.glob('*.jpg')))} images")
print(f"   new/:   {len(list(OUT_NEW.glob('*.jpg')))} images")
if missing:
    print(f"\n⚠️ {len(missing)} referenced images NOT found in their source folder:")
    print(missing[:10])

In [ ]:
# ── Cell 4: Save the final combined CSV (drop the internal tracking column) ──
final_csv = combined.drop(columns=["_source_csv"])
FINAL_CSV_PATH = Path("/kaggle/working/human_queries_FINAL_combined.csv")
final_csv.to_csv(FINAL_CSV_PATH, index=False)
print(f"✅ Saved: {FINAL_CSV_PATH}  ({len(final_csv)} rows)")

In [ ]:
# ── Cell 5: Coverage report — per-tool counts and per-tool structure breakdown ─
real_tools = ['object_detection','object_counting','person_detection','face_detection','vehicle_detection',
'animal_detection','text_detection_ocr','logo_detection','anomaly_detection','semantic_segmentation',
'instance_segmentation','panoptic_segmentation','background_removal','salient_object_segmentation','matting',
'text_to_image_generation','style_transfer','super_resolution','image_colorization','image_denoising',
'image_deblurring','depth_estimation','surface_normal_estimation','3d_reconstruction','point_cloud_generation',
'image_classification','fine_grained_classification','scene_recognition','facial_expression_recognition',
'age_gender_estimation','action_recognition','pose_estimation','gaze_estimation','visual_question_answering',
'image_captioning','visual_reasoning','visual_grounding','relation_detection','attribute_recognition',
'video_object_tracking','temporal_action_localization']

stale = set(final_csv["tool_name"]) - set(real_tools)
print(f"Stale/unrecognized tool_names: {stale}\n")

# 1. Queries per tool
per_tool_count = final_csv["tool_name"].value_counts()
full_count = pd.Series(0, index=real_tools)
full_count.update(per_tool_count)
full_count = full_count.sort_values()
full_count.to_csv("/kaggle/working/queries_per_tool.csv", header=["count"])
print("=== Queries per tool (sorted ascending) ===")
print(full_count.to_string())
print(f"\nBelow target (<10): {(full_count<10).sum()}  |  At/above target (>=10): {(full_count>=10).sum()}")

# 2. Query structure breakdown per tool
structure_crosstab = pd.crosstab(final_csv["tool_name"], final_csv["query_structure"])
structure_crosstab = structure_crosstab.reindex(real_tools, fill_value=0)
structure_crosstab.to_csv("/kaggle/working/query_structure_per_tool.csv")
print("\n=== Query structure breakdown per tool ===")
print(structure_crosstab.to_string())

In [ ]:
# ── Cell 6: Zip the final CSV, filtered image folders, and coverage reports ──
import zipfile
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M")
zip_path = Path(f"/kaggle/working/human_eval_FINAL_{ts}.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(FINAL_CSV_PATH, FINAL_CSV_PATH.name)
    zf.write("/kaggle/working/queries_per_tool.csv", "queries_per_tool.csv")
    zf.write("/kaggle/working/query_structure_per_tool.csv", "query_structure_per_tool.csv")
    for folder, label in [(OUT_KNOWN, "known"), (OUT_NEW, "new")]:
        for f in folder.glob("*.jpg"):
            zf.write(f, f"{label}/{f.name}")

print(f"✅ {zip_path.name} ({zip_path.stat().st_size/1e6:.1f} MB)")